# Wojtek: uczenie chodu krok po kroku (Colab)

Jak Wojtek uczy się chodzić: robot w MuJoCo → środowisko → trening → eksport → Twoja polityka kontra ta, która jeździ na robocie. Każdy krok kończy się widokiem z MuJoCo.

Środowisko: **GPU** (Runtime → Change runtime type). Uruchamiaj komórki po kolei.

## Krok 0 — Instalacja

Klonuje repozytorium i instaluje `training/` (JAX, MuJoCo MJX, MJWarp, Brax). Jeśli następna komórka nie zaimportuje bibliotek, zrób Runtime → Restart session i uruchom od początku.

In [ ]:
import os, subprocess, sys
from pathlib import Path

if sys.platform == "linux":
    os.environ.setdefault("MUJOCO_GL", "egl")   # renderowanie bez ekranu; przed `import mujoco`

REPO_BRANCH = "Add-notebook-with-the-guidance-how-to-train-Wojtek"   # po scaleniu: "main"
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "training" / "run.sh").exists()), None)
if REPO_ROOT is None:
    REPO_ROOT = Path.cwd() / "w01-tek"
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "-q", "-b", REPO_BRANCH, "https://github.com/machinekind/w01-tek.git", str(REPO_ROOT)], check=True)
TRAINING = REPO_ROOT / "training"

try:
    import wojtek_rl  # noqa: F401
except ImportError:
    import tomllib
    lock = tomllib.loads((TRAINING / "uv.lock").read_text())
    mujoco_pin = next(p["version"] for p in lock["package"] if p["name"] == "mujoco")   # ta sama wersja co w locku
    res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(TRAINING), f"mujoco=={mujoco_pin}"],
                         capture_output=True, text=True)
    if res.returncode:
        print(res.stdout[-1500:], res.stderr[-3000:])
        raise SystemExit("pip install nie powiodł się (patrz wyżej)")
    # Colab ma preinstalowany nowszy plugin JAX dla CUDA 13; obok jax 0.9.2 tylko generuje błędy.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "jax-cuda13-plugin", "jax-cuda13-pjrt"], capture_output=True)
    print("zainstalowano; jeśli następna komórka nie działa, zrestartuj sesję i uruchom od początku")
print(REPO_ROOT, "| python", sys.version.split()[0])

In [ ]:
import shutil
import mediapy as media
import mujoco
import numpy as np

if shutil.which("ffmpeg") is None:          # mediapy potrzebuje binarki ffmpeg
    import imageio_ffmpeg
    media.set_ffmpeg(imageio_ffmpeg.get_ffmpeg_exe())

if str(TRAINING) not in sys.path:
    sys.path.insert(0, str(TRAINING))
from wojtek_rl import paths

def show(frames, fps=25):
    media.show_video(np.asarray(frames), fps=fps, codec="h264")

print("mujoco", mujoco.__version__, "| model:", paths.SCENE_XML.relative_to(REPO_ROOT))

## Krok 1 — Wojtek stoi w MuJoCo

Model treningowy: `ros/src/wojtek_description/mujoco/scene_mjx.xml` (generowany przez `./training/run.sh build`, nie edytuj ręcznie). 12 serw pozycyjnych PD (biodro boczne, biodro, kolano × 4 nogi), nogi jako czworoboki przegubowe, 14 kg. Fizyka 250 Hz.

Robot startuje z klatki `home` i trzyma jej cele w serwach. Nic więcej: tak wygląda „zerowa akcja” polityki.

In [ ]:
model = mujoco.MjModel.from_xml_path(str(paths.SCENE_XML))
data = mujoco.MjData(model)
mujoco.mj_resetDataKeyframe(model, data, model.key("home").id)
data.ctrl[:] = model.key("home").ctrl

renderer = mujoco.Renderer(model, height=360, width=480)
frames = []
for k in range(int(4.0 / model.opt.timestep)):       # 4 s
    mujoco.mj_step(model, data)
    if k % 10 == 0:                                   # 25 klatek/s
        renderer.update_scene(data, camera="track")
        frames.append(renderer.render().copy())

print(f"serwa: {model.nu}, masa {sum(model.body_mass):.1f} kg, wysokość bazy {data.qpos[2]:.3f} m")
show(frames)